In [12]:
#All imports
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt


In [3]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving train_feature_engineered.parquet to train_feature_engineered.parquet
Saving val_feature_engineered.parquet to val_feature_engineered.parquet


In [4]:
#Loading the dataset
train = pd.read_parquet("train_feature_engineered.parquet")
val = pd.read_parquet("val_feature_engineered.parquet")

In [5]:

#All the price features which include values as medium and medium_low
valuation_features = [
    'Shillers Cyclically Adjusted P/E Ratio',
    'Enterprise Value Multiple',
    'Price/Book',
    'Price/Sales',
    'Price/Cash flow',
    'Price/Operating Earnings (Basic, Excl. EI)',
    'Price/Operating Earnings (Diluted, Excl. EI)',
    'P/E (Diluted, Excl. EI)',
    'P/E (Diluted, Incl. EI)',
    'Trailing P/E to Growth (PEG) ratio',
    'Book/Market',
    'Dividend Yield',
    'Dividend Payout Ratio'
]

# Select _Regime columns
regime_cols = [f"{col}_Regime" for col in valuation_features if f"{col}_Regime" in train.columns]

# Convert to string
for col in regime_cols:
    train[col] = train[col].astype(str)
    val[col] = val[col].astype(str)

# Missing values
for col in regime_cols:
    train[col] = train[col].replace("nan", "Missing")
    val[col] = val[col].replace("nan", "Missing")

# One-hot encoding (
train = pd.get_dummies(train, columns=regime_cols, drop_first=True)
val = pd.get_dummies(val, columns=regime_cols, drop_first=True)

# Align val and test on train columns
val = val.reindex(columns=train.columns, fill_value=0)


In [6]:
#making 3 class return
# Train
conditions_train = [
    (train['Return'] > 3),
    (train['Return'] >= -3) & (train['Return'] <= 3),
    (train['Return'] < -3)
]

choices = [1, 0, -1]

train['Return_class_3'] = np.select(conditions_train, choices, default=np.nan)


# Validation
conditions_val = [
    (val['Return'] > 3),
    (val['Return'] >= -3) & (val['Return'] <= 3),
    (val['Return'] < -3)
]

val['Return_class_3'] = np.select(conditions_val, choices, default=np.nan)



In [7]:
#making a return class
target = "Return"

X_train = train.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_train = train["Return_class_3"]

X_val = val.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_val = val["Return_class_3"]


In [8]:
#dropping columns
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])


In [9]:
#Making imputer
imputer = SimpleImputer(strategy='mean')

# Fit on training
X_train_scaled = imputer.fit_transform(X_train)

# Transform val
X_val_scaled = imputer.transform(X_val)


In [10]:
# stratified K-fold for logistic regression
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
report_list = []

fold = 1
for train_index, val_index in skf.split(X_train_scaled, y_train):
    X_tr, X_val_fold = X_train_scaled[train_index], X_train_scaled[val_index]
    y_tr, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val_fold)

    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average='macro')
    report = classification_report(y_val_fold, y_pred, output_dict=True)

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)
    report_list.append(report)

    print(f"Fold {fold} - Accuracy: {acc:.4f}, Macro F1: {macro_f1:.4f}")
    fold += 1

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 1 - Accuracy: 0.6334, Macro F1: 0.2665


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 2 - Accuracy: 0.6368, Macro F1: 0.2695


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 3 - Accuracy: 0.6312, Macro F1: 0.2659


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 4 - Accuracy: 0.6382, Macro F1: 0.2755
Fold 5 - Accuracy: 0.6310, Macro F1: 0.2600


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
# average score
print("\nAverage Accuracy:", np.mean(accuracy_list))
print("Average Macro F1:", np.mean(macro_f1_list))

# average classification report per class
avg_report = {}
classes = y_train.unique()
for cls in classes:
    avg_report[str(cls)] = {
        'precision': np.mean([r[str(cls)]['precision'] for r in report_list]),
        'recall': np.mean([r[str(cls)]['recall'] for r in report_list]),
        'f1-score': np.mean([r[str(cls)]['f1-score'] for r in report_list]),
        'support': np.mean([r[str(cls)]['support'] for r in report_list])
    }

avg_report_df = pd.DataFrame(avg_report).T
print("\nAverage Classification Report per class:")
print(avg_report_df)


Average Accuracy: 0.6341383446376037
Average Macro F1: 0.26746473390851905

Average Classification Report per class:
      precision    recall  f1-score  support
0.0    0.637063  0.991998  0.775864   1149.8
1.0    0.262210  0.005211  0.010189    345.6
-1.0   0.329221  0.008383  0.016341    310.2


In [14]:
#Important features
all_coefs = []
accuracy_list = []
macro_f1_list = []
report_list = []

fold = 1
for train_index, val_index in skf.split(X_train_scaled, y_train):
    X_tr, X_val_fold = X_train_scaled[train_index], X_train_scaled[val_index]
    y_tr, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    model.fit(X_tr, y_tr)


    fold_coefs = np.mean(np.abs(model.coef_), axis=0)
    all_coefs.append(fold_coefs)

    y_pred = model.predict(X_val_fold)

    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average='macro')
    report = classification_report(y_val_fold, y_pred, output_dict=True)

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)
    report_list.append(report)

    print(f"Fold {fold} - Accuracy: {acc:.4f}, Macro F1: {macro_f1:.4f}")
    fold += 1

mean_coefs = np.mean(all_coefs, axis=0)


feature_names = X_train.columns

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': mean_coefs
}).sort_values(by='Importance', ascending=False)

# Top 20 features
print(feature_importance.head(20))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 1 - Accuracy: 0.3184, Macro F1: 0.3222


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 2 - Accuracy: 0.3073, Macro F1: 0.3131


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 3 - Accuracy: 0.2968, Macro F1: 0.3006


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 4 - Accuracy: 0.3053, Macro F1: 0.3012
Fold 5 - Accuracy: 0.3042, Macro F1: 0.3064
                                           Feature  Importance
51                       Sales/Stockholders Equity    0.004042
49                               Payables Turnover    0.003468
27                               Total Debt/EBITDA    0.001798
107                  Sales/Stockholders Equity_QoQ    0.001574
0                                     Surprise_Pct    0.001269
34              Free Cash Flow/Operating Cash Flow    0.001187
35         Total Liabilities/Total Tangible Assets    0.001074
36                      Long-term Debt/Book Equity    0.000932
124                          Price/Book_Regime_Low    0.000813
13   After-tax Return on Total Stockholders Equity    0.000719
11       After-tax Return on Average Common Equity    0.000719
44                                   Current Ratio    0.000683
96                 After-tax Interest Coverage_QoQ    0.000682
8                             

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Statistieken voor Sales/Stockholders Equity:
count    8559.000000
mean        4.504677
std        24.096533
min         0.142559
25%         0.928399
50%         1.575791
75%         3.375533
max      1572.600000
Name: Sales/Stockholders Equity, dtype: float64
